In [1]:
!pip install ultralytics dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.

In [2]:
from google.colab import drive
from google.colab import userdata
import os
import mlflow
import dagshub

In [3]:
!mkdir /content/datasets
!unzip -o "/content/drive/MyDrive/datn/dot2/experiments/dataset/My First Project l5men.v2i.yolov8.zip" -d "/content/datasets"

Archive:  /content/drive/MyDrive/datn/dot2/experiments/dataset/My First Project l5men.v2i.yolov8.zip
 extracting: /content/datasets/valid/images/test1_frame_0069_jpg.rf.04d5f409b67041b8b6b8be32d08946d8.jpg  
 extracting: /content/datasets/valid/images/test1_frame_0185_jpg.rf.d541bd83736964e34a383bfeb936136f.jpg  
 extracting: /content/datasets/valid/images/YTDown_YouTube_Bao-cao-thi-cong-can-ho-dich-vu-kiem-soa_Media_o89YiRmjzLg_001_720p_frame_0004_jpg.rf.acbd0360ab4bda32486434a49d975b1d.jpg  
 extracting: /content/datasets/valid/images/YTDown_YouTube_cong-trinh-nha-vuon-nghi-duong-full-tho-_Media_4SQtzdss69Q_001_1080p_frame_0008_jpg.rf.39c3625c2ee0bfd9b772c4dc6207ae9c.jpg  
 extracting: /content/datasets/valid/images/YTDown_Shorts_construction-do-be-tong-lot-hang-rao_Media_P9KnAoreE0Y_001_1080p_frame_0004_jpg.rf.4aba762be3e6909dbd5a0b75f0ccffae.jpg  
 extracting: /content/datasets/valid/images/test1_frame_0005_jpg.rf.f421b2bc73b9009f6b303eabceed1f42.jpg  
 extracting: /content/dataset

In [5]:
DAGSHUB_USERNAME = "danghoanghai042"
DAGSHUB_REPO     = "ppe_v2"
DAGSHUB_TOKEN    = userdata.get('ppe_v1')
# ============================================================

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["DAGSHUB_USER_TOKEN"]       = DAGSHUB_TOKEN



In [35]:
MODEL_PATH = "mlflow-artifacts:/12789c2e2c514c84a435ecf26aec8c10/fc62e5e6431846af90522770f37206a2/artifacts/weights"
dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
)

local_path = mlflow.artifacts.download_artifacts(artifact_uri=MODEL_PATH)

Initialized MLflow to track repo "danghoanghai042/ppe_v2"

Repository danghoanghai042/ppe_v2 initialized!

In [36]:
RUN_ID = MODEL_PATH.split("/")[2]
RUN_ID

'fc62e5e6431846af90522770f37206a2'

In [37]:
from ultralytics import YOLO

model = YOLO(f'{local_path}/best.pt')

In [38]:
import torch
import time
import numpy as np

IMG_SIZE = 640
N_RUNS = 5

dummy_img = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).cuda()

print("Warming up GPU...")

model.model.eval()

for _ in range(10):
    with torch.no_grad():
        model(dummy_img)

torch.cuda.synchronize()

results = []

print("Running validation benchmark...")

all_per_class_results = []

for run in range(N_RUNS):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()

    metrics = model.val()

    torch.cuda.synchronize()
    total_time = time.time() - start_time

    speed = metrics.speed

    preprocess_ms = speed["preprocess"]
    inference_ms = speed["inference"]
    postprocess_ms = speed["postprocess"]

    latency_ms = preprocess_ms + inference_ms + postprocess_ms
    fps = 1000 / latency_ms

    results.append({
        "map50": float(metrics.box.map50),
        "map5095": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "peak_vram_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "preprocess_ms": preprocess_ms,
        "inference_ms": inference_ms,
        "postprocess_ms": postprocess_ms,
        "latency_ms": latency_ms,
        "fps": fps,
        "total_time": total_time
    })

    names = metrics.names

    for class_id, class_name in names.items():
        all_per_class_results.append({
            "run": run + 1,
            "class_id": class_id,
            "class_name": class_name,
            "precision": float(metrics.box.p[class_id]),
            "recall": float(metrics.box.r[class_id]),
            "AP50": float(metrics.box.ap50[class_id]),
            "AP50_95": float(metrics.box.ap[class_id])
        })

    print(f"Run {run + 1}/{N_RUNS} done")

def mean_std(key):
    values = np.array([r[key] for r in results])
    return values.mean(), values.std(ddof=1)

params = sum(p.numel() for p in model.model.parameters())

print("\n========== AVERAGE RESULT ==========")

for name, key, unit in [
    ("mAP50", "map50", ""),
    ("mAP50-95", "map5095", ""),
    ("Precision", "precision", ""),
    ("Recall", "recall", ""),
    ("Peak VRAM", "peak_vram_mb", " MB"),
    ("Preprocess", "preprocess_ms", " ms"),
    ("Inference", "inference_ms", " ms"),
    ("Postprocess", "postprocess_ms", " ms"),
    ("Latency", "latency_ms", " ms"),
    ("FPS", "fps", ""),
    ("Total Time", "total_time", " sec"),
]:
    mean, std = mean_std(key)
    print(f"{name:<12}: {mean:.4f} ± {std:.4f}{unit}")

print(f"Params      : {params:,}")
print("====================================")

Warming up GPU...

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.359316825866699. Dividing input by 255.
0: 640x640 (no detections), 83.0ms
Speed: 0.1ms preprocess, 83.0ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.359316825866699. Dividing input by 255.
0: 640x640 (no detections), 72.0ms
Speed: 0.0ms preprocess, 72.0ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.359316825866699. Dividing input by 255.
0: 640x640 (no detections), 61.7ms
Speed: 0.0ms preprocess, 61.7ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.359316825866699. Dividing input by 255.
0: 640x640 (no detections), 57.8ms
Speed: 0.1ms preprocess, 57.8ms inference, 1.1ms postprocess per image 

In [39]:
import os
from mlflow.tracking import MlflowClient

metrics = [
    ("mAP50", "map50", ""),
    ("mAP50-95", "map5095", ""),
    ("Precision", "precision", ""),
    ("Recall", "recall", ""),
    ("Peak VRAM", "peak_vram_mb", " MB"),
    ("Preprocess", "preprocess_ms", " ms"),
    ("Inference", "inference_ms", " ms"),
    ("Postprocess", "postprocess_ms", " ms"),
    ("Latency", "latency_ms", " ms"),
    ("FPS", "fps", ""),
    ("Total Time", "total_time", " sec"),
]

rows = []
client = MlflowClient()

for name, key, unit in metrics:
    mean, std = mean_std(key)

    print(f"{name:<12}: {mean:.4f} ± {std:.4f}{unit}")

    client.log_metric(
        run_id=RUN_ID,
        key=f"eval/{key}_mean_{N_RUNS}",
        value=float(mean)
    )

    client.log_metric(
        run_id=RUN_ID,
        key=f"eval/{key}_std_{N_RUNS}",
        value=float(std)
    )

    rows.append({
        "metric": name,
        "key": key,
        "mean": float(mean),
        "std": float(std),
        "unit": unit.strip(),
        "n_runs": N_RUNS,
    })

df_metrics = pd.DataFrame(rows)

os.makedirs("mlflow_outputs", exist_ok=True)
csv_path = "mlflow_outputs/eval_metrics.csv"

df_metrics.to_csv(csv_path, index=False)

client.log_artifact(
    run_id=RUN_ID,
    local_path=csv_path,
    artifact_path="eval"
)

confusion_matrix_path = "/content/runs/detect/val-5/confusion_matrix.png"


client.log_artifact(
    run_id=RUN_ID,
    local_path=confusion_matrix_path,
    artifact_path="eval"
)

print(f"Params: {params:,}")
print(f"Saved metrics CSV: {csv_path}")

mAP50       : 0.6422 ± 0.0000
mAP50-95    : 0.4098 ± 0.0000
Precision   : 0.6691 ± 0.0000
Recall      : 0.6241 ± 0.0000
Peak VRAM   : 2197.9395 ± 233.6515 MB
Preprocess  : 1.8455 ± 0.4834 ms
Inference   : 37.4016 ± 1.1848 ms
Postprocess : 0.2252 ± 0.0421 ms
Latency     : 39.4723 ± 0.9199 ms
FPS         : 25.3453 ± 0.5943
Total Time  : 23.1385 ± 1.0775 sec
Params: 32,002,235
Saved metrics CSV: mlflow_outputs/eval_metrics.csv


In [40]:
import pandas as pd
per_class_df = pd.DataFrame(all_per_class_results)

per_class_summary = (
    per_class_df
    .groupby(["class_id", "class_name"])
    .agg({
        "precision": ["mean"],
        "recall": ["mean"],
        "AP50": ["mean"],
        "AP50_95": ["mean"]
    })
    .reset_index()
)

per_class_summary.to_csv('perclass_v2.csv', index=False)

In [41]:
perclass_path = '/content/perclass_v2.csv'
client.log_artifact(
    run_id=RUN_ID,
    local_path=perclass_path,
    artifact_path="eval"
)

In [ ]:
# %cd /content/runs/detect
# !zip -r val